<a href="https://colab.research.google.com/github/arelkeselbri/pgc305/blob/main/aula0_1_regressao_logistica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PGC305 - Tópicos especiais em LLM e Deep Learning

## Definição dos dados

In [42]:
import torch; import sklearn
from sklearn.model_selection import train_test_split

# 1. Carregar dados
iris = sklearn.datasets.load_iris()

X = iris.data        # 4 features: sépalas e pétalas
y = (iris.target == 1).astype(float)  # 1 se Versicolor, 0 caso contrário

# 1. Separar dados de Treino e um conjunto teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42 # 70% treino, 30% temp
)

# 2. Preparar dados para pytorch
X = torch.tensor(X_train, dtype=torch.float32)  # (150, 4) -> 150 amostras, 4 features
y = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)


## Definição do modelo e treinamento

In [43]:
# 3. Definir modelo: regressão logística
modelo = torch.nn.Linear(4, 1)  # 4 features → 1 saída (probabilidade de ser Versicolor)

# 4. Definir função de perda e algoritmo de otimização
#funcao_perda = torch.nn.BCEWithLogitsLoss()  # combinação de sigmoid + BCE
funcao_perda = torch.nn.BCEWithLogitsLoss()  # combinação de sigmoid + BCE
optimizer = torch.optim.SGD(modelo.parameters(), lr=0.1)

## Execução do treinamento

In [44]:
# 5. Treino
for epoch in range(1000):
    optimizer.zero_grad() # reseta gradiente senão acumula
    outputs = modelo(X)
    loss = funcao_perda(outputs, y)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Época [{epoch+1}/100], Loss: {loss.item():.4f}")

Época [10/100], Loss: 0.6994
Época [20/100], Loss: 0.6652
Época [30/100], Loss: 0.6487
Época [40/100], Loss: 0.6392
Época [50/100], Loss: 0.6327
Época [60/100], Loss: 0.6277
Época [70/100], Loss: 0.6235
Época [80/100], Loss: 0.6196
Época [90/100], Loss: 0.6161
Época [100/100], Loss: 0.6127
Época [110/100], Loss: 0.6095
Época [120/100], Loss: 0.6065
Época [130/100], Loss: 0.6036
Época [140/100], Loss: 0.6007
Época [150/100], Loss: 0.5980
Época [160/100], Loss: 0.5954
Época [170/100], Loss: 0.5929
Época [180/100], Loss: 0.5905
Época [190/100], Loss: 0.5882
Época [200/100], Loss: 0.5860
Época [210/100], Loss: 0.5839
Época [220/100], Loss: 0.5818
Época [230/100], Loss: 0.5798
Época [240/100], Loss: 0.5779
Época [250/100], Loss: 0.5761
Época [260/100], Loss: 0.5743
Época [270/100], Loss: 0.5726
Época [280/100], Loss: 0.5709
Época [290/100], Loss: 0.5693
Época [300/100], Loss: 0.5678
Época [310/100], Loss: 0.5663
Época [320/100], Loss: 0.5649
Época [330/100], Loss: 0.5635
Época [340/100], Lo

### Avaliação se o modelo aprendido esta bom

Estava com 70% de accuracy, um valor razoavel

Dividindo o dataset 70/30 deu 73% 

In [46]:
pesos = modelo.weight
bias = modelo.bias

modelo.eval()

X = torch.tensor(X_test, dtype=torch.float32)
y = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

outputs = modelo(X)
predicted = (torch.sigmoid(outputs) > 0.5).float()  # classifica como 1 se prob > 0.5
accuracy = (predicted == y).float().mean()
print(f"Acurácia: {accuracy.item():.4f}")


Acurácia: 0.7333


In [ ]:
# identificar o que foi aprendido no nn.linear

modelo.forward(X)

tensor([[ 0.1310],
        [-2.0209],
        [ 1.0507],
        [-0.5615],
        [ 0.4204],
        [-1.6231],
        [-0.8938],
        [-1.0854],
        [ 0.9705],
        [-0.1391],
        [-1.2089],
        [-0.9275],
        [-1.5753],
        [-1.0232],
        [-2.5106],
        [-1.1986],
        [-0.8272],
        [ 0.2358],
        [-0.3365],
        [-0.5650],
        [-1.4513],
        [-0.9202],
        [-1.8749],
        [-0.4453],
        [-0.9912],
        [-0.9968],
        [ 0.7862],
        [-1.0992],
        [-1.1668],
        [-1.1818],
        [-2.5085],
        [-3.3960],
        [-0.3742],
        [-1.7775],
        [-1.7602],
        [ 0.1269],
        [-0.8734],
        [-1.7243],
        [-2.0967],
        [-2.7961],
        [-0.5930],
        [-1.6740],
        [-0.3980],
        [-2.6799],
        [-1.9795]], grad_fn=<AddmmBackward0>)

4 - Colocar 3 saidas
 - Alterar o nn.Linear para 3 outputs (probabilidade de cada classe)
 - Cross entropy Loss (Classificação multi-classe)

In [ ]:
# 1. Carregar dados
iris = sklearn.datasets.load_iris()

X = iris.data        # 4 features: sépalas e pétalas
y = (iris.target == 1).astype(float)  # 1 se Versicolor, 0 caso contrário

# 2. Preparar dados para pytorch
X = torch.tensor(X, dtype=torch.float32)  # (150, 4) -> 150 amostras, 4 features
shapeX = X.shape  # verificar forma de X
y = torch.tensor(y, dtype=torch.float32).view(-1, 1)
shapeY = y.dim()  # verificar forma de y

In [9]:
# 3. Definir modelo: regressão logística
modelo = torch.nn.Linear(4, 3)  # 4 features → 3 saídas (probabilidades de ser cada classe)

# 4. Definir função de perda e algoritmo de otimização
funcao_perda = torch.nn.CrossEntropyLoss()  # função de perda alterada para classificação multi-classe
optimizer = torch.optim.SGD(modelo.parameters(), lr=0.1)

In [10]:
# 5. Treino - cross entropy espera rótulos como inteiros (0, 1, 2)
for epoch in range(1000):
    optimizer.zero_grad() # reseta gradiente senão acumula
    outputs = modelo(X)
    loss = funcao_perda(outputs, y)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Época [{epoch+1}/100], Loss: {loss.item():.4f}")

RuntimeError: 0D or 1D target tensor expected, multi-target not supported